In [4]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import string

In [5]:
train_df = pd.read_csv("../data/train_translated_tokenized.csv")
train_df.rename(columns={train_df.columns[0]: "real_dataset_index"}, inplace=True)


In [6]:
def generate_n_gram_table(
    df: pd.DataFrame,
    n: int = 2,
    normalize: bool = True,
) -> pd.DataFrame:
    n_gram_counter = Counter()

    for tokens in df:
        tokens = ["<START>"] * (n - 1) + tokens + ["<END>"] * (n - 1)
        n_gram_counter.update(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))
    
    df_matrix = pd.DataFrame(
        [(ngram[:-1], ngram[-1], count) for ngram, count in n_gram_counter.items()],
        columns=["prefix", "next_word", "count"]
    )
    df_matrix["prefix"] = df_matrix["prefix"].apply(lambda x: " ".join(x))
    
    pivot_table = df_matrix.pivot_table(
        index="prefix",
        columns="next_word",
        values="count",
        fill_value=0
    )
    
    pivot_table["<UNK>"] = 0
    if n >= 2:
        unk_row = pd.Series(0, index=pivot_table.columns, name="<UNK>")
        pivot_table = pd.concat([pivot_table, unk_row.to_frame().T])


    pivot_table = pivot_table + 1
    
    if normalize:
        pivot_table = pivot_table.div(pivot_table.sum(axis=1), axis=0)
    
    return pivot_table


In [7]:
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

In [9]:
lang="ko"
lang_df = train_df[train_df["lang"] == lang].copy()

lang_df["question_stripped"] = lang_df["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

korean_bigram = generate_n_gram_table(lang_df["question_stripped"], n=2)

In [10]:
korean_bigram.to_parquet("../data/bi_grams/korean_bigram.parquet")

In [11]:
lang="te"
lang_df = train_df[train_df["lang"] == lang].copy()

lang_df["question_stripped"] = lang_df["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

telugu_bigram = generate_n_gram_table(lang_df["question_stripped"], n=2)

In [13]:
telugu_bigram.to_parquet("../data/bi_grams/telugu_bigram.parquet")

In [15]:
lang="ar"
lang_df = train_df[train_df["lang"] == lang].copy()

lang_df["question_stripped"] = lang_df["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

arabic_bigram = generate_n_gram_table(lang_df["question_stripped"], n=2)

In [17]:
arabic_bigram.to_parquet("../data/bi_grams/arabic_bigram.parquet")

In [6]:

context_series = train_df["context_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

context_bigram = generate_n_gram_table(context_series, n=2)

/var/folders/qr/281wf9617dgbt8dprdwtfcyr0000gn/T/ipykernel_48819/2627621237.py:18: PerformanceWarning: The following operation may generate 2216338084 cells in the resulting pandas object.
  pivot_table = df_matrix.pivot_table(


In [ ]:
context_bigram.to_parquet("../data/bi_grams/context_bigram.parquet", index=True)

In [14]:
value = context_bigram.loc["한국과학기술연구원", "<UNK>"]
print(value)

2.1240441801189464e-05
